In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week5_Data_Processing") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)

Spark Version: 3.5.1


## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce processes data by reading from and writing intermediate results to disk after each stage, which makes it slower for iterative and real-time applications. It also requires more code for complex operations and has higher latency. Apache Spark overcomes these limitations through in-memory processing, faster execution, easy-to-use APIs, fault tolerance, and support for machine learning, streaming, graph processing, and interactive analytics.

In [2]:
data = [
    (1, "Alice", "alice@gmail.com", 25, "Premium", "West", "Electronics", 1200.0, "2026-07-01", "New York"),
    (2, "Bob", None, 32, "Basic", "East", "Furniture", 850.0, "2026-07-02", "Boston"),
    (3, "", "charlie@gmail.com", 28, "Premium", "West", "Electronics", None, "2026-07-03", "Seattle"),
    (4, "David", "david@gmail.com", 22, "Premium", "South", "Clothing", 500.0, "2026-07-04", "Dallas"),
    (5, "Emma", "emma@gmail.com", 19, "Premium", "West", "Furniture", 750.0, "2026-07-05", "Seattle"),
    (5, "Emma", "emma@gmail.com", 19, "Premium", "West", "Furniture", 750.0, "2026-07-05", "Seattle")
]

columns = [
    "user_id",
    "username",
    "email",
    "age",
    "subscription",
    "region",
    "product_category",
    "price",
    "transaction_date",
    "city"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|user_id|username|            email|age|subscription|region|product_category| price|transaction_date|    city|
+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|      1|   Alice|  alice@gmail.com| 25|     Premium|  West|     Electronics|1200.0|      2026-07-01|New York|
|      2|     Bob|             NULL| 32|       Basic|  East|       Furniture| 850.0|      2026-07-02|  Boston|
|      3|        |charlie@gmail.com| 28|     Premium|  West|     Electronics|  NULL|      2026-07-03| Seattle|
|      4|   David|  david@gmail.com| 22|     Premium| South|        Clothing| 500.0|      2026-07-04|  Dallas|
|      5|    Emma|   emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
|      5|    Emma|   emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
+

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate data in memory (RAM) instead of writing it to disk after every computation. Since machine learning algorithms repeatedly process the same dataset over multiple iterations, keeping the data in memory significantly reduces disk I/O and improves execution speed. This makes Spark much faster and more efficient than traditional disk-based systems like MapReduce.

In [4]:
# Remove duplicate rows based on user_id and transaction_date

df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])

df_no_duplicates.show()

+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|user_id|username|            email|age|subscription|region|product_category| price|transaction_date|    city|
+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|      1|   Alice|  alice@gmail.com| 25|     Premium|  West|     Electronics|1200.0|      2026-07-01|New York|
|      2|     Bob|             NULL| 32|       Basic|  East|       Furniture| 850.0|      2026-07-02|  Boston|
|      3|        |charlie@gmail.com| 28|     Premium|  West|     Electronics|  NULL|      2026-07-03| Seattle|
|      4|   David|  david@gmail.com| 22|     Premium| South|        Clothing| 500.0|      2026-07-04|  Dallas|
|      5|    Emma|   emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+



In [5]:
# Filter rows where region is 'West' and find average price by product category

west_sales = df.filter(col("region") == "West") \
               .groupBy("product_category") \
               .agg(avg("price").alias("average_sale_amount"))

west_sales.show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|     Electronics|             1200.0|
|       Furniture|              750.0|
+----------------+-------------------+



## Q5. Difference between `.na.drop()` and `.na.fill()`

- **`.na.drop()`** removes rows that contain null values.
- **`.na.fill()`** replaces null values with a specified value without removing the rows.

Example:

```python
df.na.fill({"status": "Unknown"})
```

In [6]:
# Count records for each city and display only cities with count > 100

city_counts = df.groupBy("city") \
                .count() \
                .filter(col("count") > 100)

city_counts.show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



## Q7. How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

Spark DataFrames are immutable, which means they cannot be modified after they are created. Any transformation such as dropping columns, renaming columns, or filtering data creates a new DataFrame instead of changing the original one. This ensures data consistency, fault tolerance, and easier debugging.

In [7]:
# Filter users aged between 18 and 30 with Premium subscription

premium_users = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

premium_users.show()

+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|user_id|username|            email|age|subscription|region|product_category| price|transaction_date|    city|
+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+
|      1|   Alice|  alice@gmail.com| 25|     Premium|  West|     Electronics|1200.0|      2026-07-01|New York|
|      3|        |charlie@gmail.com| 28|     Premium|  West|     Electronics|  NULL|      2026-07-03| Seattle|
|      4|   David|  david@gmail.com| 22|     Premium| South|        Clothing| 500.0|      2026-07-04|  Dallas|
|      5|    Emma|   emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
|      5|    Emma|   emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
+-------+--------+-----------------+---+------------+------+----------------+------+----------------+--------+



## Q9. Why is it better to handle null values before performing mathematical aggregations like sum() or avg()?

Handling null values before performing aggregations ensures accurate and reliable results. Null values can lead to incorrect calculations, missing values in the output, or unexpected behavior during data analysis. Cleaning the data beforehand improves data quality and the correctness of analytical results.

In [8]:
from pyspark.sql.functions import to_timestamp

timestamp_df = spark.createDataFrame(
    [("2026-07-01 10:30:00",)],
    ["raw_timestamp"]
)

timestamp_df = timestamp_df.withColumn(
    "event_time",
    to_timestamp(col("raw_timestamp"))
).drop("raw_timestamp")

timestamp_df.show(truncate=False)

+-------------------+
|event_time         |
+-------------------+
|2026-07-01 10:30:00|
+-------------------+



## Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Shuffle is the process of redistributing data across different partitions so that records with the same key are grouped together. It occurs during operations such as `groupBy()`, `join()`, and `reduceByKey()`. Shuffle is considered a wide transformation because it requires data movement between partitions, which increases network communication and execution time.

In [9]:
# Remove rows with null email or empty username

clean_df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

clean_df.show()

+-------+--------+---------------+---+------------+------+----------------+------+----------------+--------+
|user_id|username|          email|age|subscription|region|product_category| price|transaction_date|    city|
+-------+--------+---------------+---+------------+------+----------------+------+----------------+--------+
|      1|   Alice|alice@gmail.com| 25|     Premium|  West|     Electronics|1200.0|      2026-07-01|New York|
|      4|   David|david@gmail.com| 22|     Premium| South|        Clothing| 500.0|      2026-07-04|  Dallas|
|      5|    Emma| emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
|      5|    Emma| emma@gmail.com| 19|     Premium|  West|       Furniture| 750.0|      2026-07-05| Seattle|
+-------+--------+---------------+---+------------+------+----------------+------+----------------+--------+



In [10]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+-------------+
|Minimum Price|Maximum Price|Average Price|
+-------------+-------------+-------------+
|        500.0|       1200.0|        810.0|
+-------------+-------------+-------------+



## Q14. What is the risk of using `inferSchema=true` when source data contains inconsistent date formats?

When `inferSchema=true` is used on data with inconsistent date formats, Spark may incorrectly infer the data type or treat the column as a string instead of a date. This can lead to parsing errors, incorrect calculations, and inconsistent query results. Defining the schema explicitly helps ensure data accuracy and consistency.

In [11]:
# Complete processing pipeline

final_df = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("city")
      .agg(sum("price").alias("Total Revenue"))
)

final_df.show()

+--------+-------------+
|    city|Total Revenue|
+--------+-------------+
|  Dallas|        500.0|
| Seattle|        750.0|
|New York|       1200.0|
|  Boston|        850.0|
+--------+-------------+

